<a id="top"></a>

# Demo: few-shot is a precision tool, not a default

```yaml
title: "Demo: few-shot is a precision tool, not a default"
keywords: few-shot, in-context examples, diversity, overfitting, placement, cost, teacher demo
estimated duration: 20
```

> **Lesson:** L02. This is Demo 3 in the [roadmap](../../../../docs/origin/lesson_roadmaps/L02/demos_or_activities.md). It makes **live** Claude calls through the `potato_llm` seam — set `ANTHROPIC_API_KEY` (copy `.env.example` to `.env`) before running. A real model **varies** run to run, so exact predictions below may differ; watch for the *pattern* each section calls out. Clear outputs before committing.
>
> The task: classify tickets into a team's **idiosyncratic** taxonomy the model wouldn't know from training.

## Contents

- [1. Setup](#1-setup)
- [2. Instruction-only fails on the custom labels](#2-instruction-only-fails-on-the-custom-labels)
- [3. Uniform examples overfit](#3-uniform-examples-overfit)
- [4. Diverse examples, and two placements](#4-diverse-examples-and-two-placements)
- [5. The cost of few-shot, and the editable trap](#5-the-cost-of-few-shot-and-the-editable-trap)
- [6. Takeaways](#6-takeaways)

## 1. Setup

You've got one live client and five test tickets, each with the team's true (idiosyncratic) label. Across the sections you'll change only the *examples* you plant — none, uniform, or diverse — and watch the predictions shift.

In [ ]:
from fluffy_potato_curriculum.potato_llm import AnthropicClient, Message, PotatoLLMClient

LABELS = ["P0-bug", "feature-discussion", "docs-clarity", "billing-question", "thank-you-note"]

# Five test tickets and their TRUE team label (the team's idiosyncratic wording).
TICKETS: list[tuple[str, str]] = [
    ("The checkout page returns a 500 error for every user since the deploy.", "P0-bug"),
    ("I was charged twice for my subscription this month.", "billing-question"),
    ("The README example uses parameter names that don't match the actual API.", "docs-clarity"),
    (
        "Could we discuss adding dark mode? The docs also don't mention theming.",
        "feature-discussion",
    ),
    ("Just wanted to say your support team is amazing — thank you so much!", "thank-you-note"),
]

INSTRUCTION = (
    "Classify the ticket into exactly one of: " + ", ".join(LABELS) + ". Return only the label."
)

# Live client. The key is read through fluffy_potato_curriculum.common.config
# (a pydantic-settings layer over the environment / .env); constructing the client
# raises a clear error if ANTHROPIC_API_KEY is missing, rather than failing mid-call.
client: PotatoLLMClient = AnthropicClient()
print(f"setup ready (live client: {client.model})")

[↑ Back to top](#top)

## 2. Instruction-only fails on the custom labels

Here's a clear instruction, no examples. Watch how the model can't guess the team's exact label wording.

In [ ]:
async def classify(prefix: list[Message], ticket: str) -> str:
    """Send the example prefix + the real ticket; return the predicted label."""
    messages = [*prefix, Message.user(f"Classify this ticket: {ticket}")]
    return (await client.achat(messages, max_tokens=8, temperature=0.0)).text


instruction_only: list[Message] = [Message.system(INSTRUCTION)]
for ticket, true in TICKETS:
    got = await classify(instruction_only, ticket)
    flag = "ok " if got == true else "MISS"
    print(f"[{flag}] got {got!r:20} (true: {true})")

Instruction-only, the model can't know *this team's* exact wording — expect near-misses like `bug` for `P0-bug`, or `positive feedback` for `thank-you-note`. The instruction is clear; the **convention** is idiosyncratic, and nothing in the prompt teaches it to the model.

[↑ Back to top](#top)

## 3. Uniform examples overfit

Now add four examples — but all of them `P0-bug`. Watch for the overfit failure mode.

In [ ]:
uniform: list[Message] = [
    Message.system(INSTRUCTION),
    Message.user("Classify this ticket: Login throws a 500 for everyone."),
    Message.assistant("P0-bug"),
    Message.user("Classify this ticket: The app crashes on startup after the update."),
    Message.assistant("P0-bug"),
    Message.user("Classify this ticket: Payments fail with a 500 at checkout."),
    Message.assistant("P0-bug"),
    Message.user("Classify this ticket: API returns 500 on every request."),
    Message.assistant("P0-bug"),
]
for ticket, true in TICKETS:
    got = await classify(uniform, ticket)
    flag = "ok " if got == true else "MISS"
    print(f"[{flag}] got {got!r:20} (true: {true})")

With every example labeled `P0-bug`, the model gets pulled toward calling **everything** a `P0-bug`. You may not get a clean sweep on every run, but the bias toward the over-represented label is the **all-examples-look-like-the-easy-case** failure: a uniform set teaches a surface pattern, not the task.

[↑ Back to top](#top)

## 4. Diverse examples, and two placements

Now try four examples covering four *different* labels. First, as alternating fake `user`/`assistant` turns.

In [ ]:
diverse_turns: list[Message] = [
    Message.system(INSTRUCTION),
    Message.user("Classify this ticket: Login throws a 500 for everyone."),
    Message.assistant("P0-bug"),
    Message.user("Classify this ticket: Can we add CSV export to reports?"),
    Message.assistant("feature-discussion"),
    Message.user("Classify this ticket: The setup guide skips the env-var step."),
    Message.assistant("docs-clarity"),
    Message.user("Classify this ticket: My invoice shows the wrong amount."),
    Message.assistant("billing-question"),
]
for ticket, true in TICKETS:
    got = await classify(diverse_turns, ticket)
    flag = "ok " if got == true else "MISS"
    print(f"[{flag}] got {got!r:20} (true: {true})")

With four labels demonstrated, expect **most** predictions to land on the team's exact wording now. The likely miss is the **off-distribution** `thank-you-note` — none of the four examples is a thank-you, so the model reaches for the nearest demonstrated label. Hold that thought for section 5.

Now try the same four examples, as a **single block** inside one user message. Compare the result.

In [ ]:
block = (
    INSTRUCTION
    + "\n\nExamples:\n"
    + "\n".join(
        [
            "Login throws a 500 for everyone. -> P0-bug",
            "Can we add CSV export to reports? -> feature-discussion",
            "The setup guide skips the env-var step. -> docs-clarity",
            "My invoice shows the wrong amount. -> billing-question",
        ]
    )
)
diverse_block: list[Message] = [Message.system(block)]
for ticket, true in TICKETS:
    got = await classify(diverse_block, ticket)
    flag = "ok " if got == true else "MISS"
    print(f"[{flag}] got {got!r:20} (true: {true})")

You should see similar quality to the alternating-turn placement. **Placement matters less than content** — pick whichever your parser or cache strategy prefers.

[↑ Back to top](#top)

## 5. The cost of few-shot, and the editable trap

Few-shot is paid for on **every** call. Compare the input size of each variant.

In [ ]:
def prefix_tokens(prefix: list[Message]) -> int:
    """Rough input-token estimate for an example prefix (~4 chars/token)."""
    return sum(len(m.content) for m in prefix) // 4


print(f"instruction-only : ~{prefix_tokens(instruction_only):4} input tok / call")
print(f"diverse (turns)  : ~{prefix_tokens(diverse_turns):4} input tok / call")
print(f"diverse (block)  : ~{prefix_tokens(diverse_block):4} input tok / call")

Now fix the off-distribution miss by **adding a fifth example** that resembles it — a thank-you note. Few-shot is *editable*: every failure you find, you can patch with an example.

In [ ]:
diverse_plus = [
    *diverse_turns,
    Message.user("Classify this ticket: You all are wonderful, thanks a ton!"),
    Message.assistant("thank-you-note"),
]
off_ticket, off_true = TICKETS[4]
print(
    "before adding the example:", await classify(diverse_turns, off_ticket), f"(true: {off_true})"
)
print("after  adding the example:", await classify(diverse_plus, off_ticket), f"(true: {off_true})")
print(f"...but the prefix grew to ~{prefix_tokens(diverse_plus)} input tok / call")

That is the power **and** the trap: the example list grows, and so does the per-call cost. Past some point you'll want to reach for a different tool — retrieval (L21), a different model class (L14), or fine-tuning (out of scope). Few-shot is a **precision tool, not a default**.

[↑ Back to top](#top)

## 6. Takeaways

- Few-shot is **behavior conditioning, not teaching** — it shifts the in-context distribution, not the model's weights.
- **Diversity beats volume**: four examples over four labels beat four of the same label.
- **Placement matters less than content** — alternating turns ≈ single block.
- Few-shot is **paid for on every call** (L01) and is **editable** — power and trap.
- Heading to L06: chain-of-thought few-shot adds a *reasoning trace* to each example. Same technique, applied to thinking — you'll get there in L06.

[↑ Back to top](#top)